## Prerequisites

Runtime: Python 3, T4 GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Pin transformers to 4.46.3:
#   - Phi-3.5-vision support was added in 4.43.0, so 4.46.3 is fine.
#   - 4.46.3 still has DynamicCache.from_legacy_cache (removed in 4.48).
#   - 4.46.3 calls available_devices.discard() only after ensuring it's a set,
#     avoiding the frozenset AttributeError introduced by bitsandbytes >=0.45.
# flash_attn is optional — we use _attn_implementation='eager' to skip it entirely.
%pip install -q "transformers==4.46.3" accelerate bitsandbytes Pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 61.2 MB/s eta 0:00:00


In [3]:
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

In [4]:
import torch

In [5]:
from pathlib import Path
from PIL import Image

WORKING_DIR = Path('/content/drive/MyDrive/aiOCR')
MODEL_NAME = 'microsoft/Phi-3.5-vision-instruct'

# Phi-3.5-vision-instruct is 4.2B params → ~2.5 GB at 4-bit, comfortable on T4's 15 GB.
qc = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

# num_crops=4: number of image crop tiles sent to the vision encoder.
# Higher values capture more fine detail but cost more VRAM.
# 4 is a safe value for T4 alongside 4-bit weights.
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    num_crops=4,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=qc,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
    _attn_implementation='eager',  # Flash Attention 2 requires Ampere+ (T4 is Turing)
).eval()

processor_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

processing_phi3_v.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-vision-instruct:
- processing_phi3_v.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:520: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/442 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.35G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

In [6]:
IMAGE_FILE = WORKING_DIR / 'images/pineda1/pineda1_page_4.png'

## Inference

In [7]:
import time

image_stem   = IMAGE_FILE.stem
image_folder = IMAGE_FILE.parent.name

image = Image.open(IMAGE_FILE).convert('RGB')

prompt = (
    'Convert the document to plain text, as close to the original as possible '
    '(including typos, print errors, and original grammar and spelling). '
    'Do not add any formatting, markdown, or annotations.'
)

# Phi-3.5-vision uses <|image_1|> placeholder tokens in the chat template.
messages = [
    {'role': 'user', 'content': '<|image_1|>\n' + prompt},
]

text = processor.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = processor(text=text, images=[image], return_tensors='pt').to(model.device)
input_len = inputs['input_ids'].shape[-1]

t0 = time.time()
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=4096,
        do_sample=False,   # greedy — avoids multinomial NaN issues on T4 fp16
    )
elapsed = time.time() - t0

generated_ids_trimmed = [out[input_len:] for out in generated_ids]
transcription = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0]

print(f'Done in {elapsed:.1f}s')
print(transcription)

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48


Done in 359.5s


puede asegurar la sentencia que ha de entregar los inocentes.
Escapenic en el rostro, le aboefetan, le juzgaron con verdades
hastes dejar descubertas las venenas de los pies en la cabeza.
La acredidad de la verdadera insultante. Como el
tigre que juega con un preso, antes de devorar, así
aquel pueblo hérobaba una antigua mascota de cordero
vérter su sangre. Le vistieron una tinta de escarpe: le
ponen en la manto una calla de gúisis de cierto an
dándole los ojos en el senillo de la diadena: luego ven-
diólo los ojos dblan en rodilla, le dan fuerzas bofetadas
en el rostro i le dican: Dios te guarda, reí de los judíos.
Entre aquel pueblo de vedgos no se hallaría uno que
no hubiese experimentado los salubres efectos de la
poderosa bofetada de él, en su pura y en la de los suyos.
Purificó a los los lejos de los restijos, la vista a los ciegos i el
mundo, a todos hice bí en el indemnizados, rescúto los
muertos, a todos hice bí a los niños, con el cúmulo
conmico como un vaso 

### Saving the output

In [8]:
# Transcription → transcriptions/Phi-3.5-vision/<stem>.md
transcription_out = WORKING_DIR / 'transcriptions/Phi-3.5-vision'
transcription_out.mkdir(parents=True, exist_ok=True)
(transcription_out / f'{image_stem}.md').write_text(transcription, encoding='utf-8')
print(f'Saved: transcriptions/Phi-3.5-vision/{image_stem}.md')

Saved: transcriptions/Phi-3.5-vision/pineda1_page_4.md
